#### Instead of standard absa and making the model extract the aspects and give a sentiment here the aspects are given but the model needs to give a sentiment conditioned Aspect Based Sentiment Analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()



In [ ]:
def generate_chat(model, tokenizer, user_content: str):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,      # deterministic
            temperature=None,     # ignored when do_sample=False
            top_p=None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
import json
def extract_first_json(text):
    if not text:
        return None

    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                block = text[start:i+1]
                try:
                    return json.loads(block)
                except:
                    return None
    return None

##### Changed the Extract First Json because it couldnt track the {...} block because it was nested in here

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev = load_jsonl(absa_dev_path)
absa_test = load_jsonl(absa_test_path)

In [ ]:
def flatten_absa_rows(rows):
    flat = []
    for r in rows:
        for lab in r["labels"]:
            flat.append({
                "text": r["text"],
                "category": lab["category"],
                "polarity": lab["polarity"].lower()
            })
    return flat

absa_train_flat = flatten_absa_rows(absa_train)

In [ ]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}
ABSA_POLARITY_SET = set(ABSA_POLARITIES)

In [ ]:
from collections import defaultdict

def pick_demos_balanced_absa(rows, shots):
    buckets = defaultdict(list)
    for r in rows:
        buckets[r["polarity"]].append(r)

    if shots == 3:
        plan = {"positive": 1, "negative": 1, "neutral": 1}
    elif shots == 5:
        plan = {"positive": 2, "negative": 1, "neutral": 1, "conflict": 1}
    else:
        raise ValueError("shots must be 3 or 5")

    demos = []
    for pol, k in plan.items():
        demos.extend(buckets[pol][:k])

    return demos

demos_3 = pick_demos_balanced_absa(absa_train_flat, 3)
demos_5 = pick_demos_balanced_absa(absa_train_flat, 5)

In [ ]:
def format_absa_demos(demos):
    lines = ["أمثلة (Examples):"]
    for d in demos:
        lines.append(f'category: {d["category"]}')
        lines.append(f'text: {d["text"]}')
        lines.append(f'output: {{"labels":[{{"polarity":"{d["polarity"]}"}}]}}')
        lines.append("")
    return "\n".join(lines).strip()

def build_conditioned_absa_prompt(text, category, demos=None):
    header = (
        "أنت نظام تصنيف فقط، وليس مساعداً.\n"
        "مهمتك الوحيدة هي اختيار قيمة polarity صحيحة.\n\n"

        "سيتم إعطاؤك:\n"
        "- نص مراجعة فندق واحد\n"
        "- جانب واحد فقط (category)\n\n"

        "المطلوب:\n"
        "اختيار polarity هذا الجانب فقط.\n\n"

        "القيم المسموحة حصراً (اكتبها كما هي بالإنجليزية):\n"
        "positive | negative | neutral | conflict\n\n"

        "قواعد إلزامية صارمة:\n"
        "1) لا تكتب أي شرح أو تفسير أو تعليق.\n"
        "2) لا تذكر النص مرة أخرى.\n"
        "3) لا تذكر اسم الـ category.\n"
        "4) لا تستخدم كلمات عربية للمشاعر.\n"
        "5) لا تضف أي نص خارج JSON.\n"
        "6) الإخراج يجب أن يكون JSON فقط وعلى سطر واحد.\n"
        "7) يجب أن يبدأ الإخراج بـ { وينتهي بـ }.\n"
        "8) أي مخالفة لهذه القواعد تعتبر خطأ.\n\n"

        "صيغة الإخراج المطلوبة حرفياً:\n"
        "{\"labels\":[{\"polarity\":\"positive\"}]}\n\n"
    )

    demo_block = (format_absa_demos(demos) + "\n\n") if demos else ""

    return (
        header +
        demo_block +
        f"category: {category}\n"
        f"text: {text}\n\n"
        "output:\n"
    )

In [ ]:
def parse_polarity_from_obj(obj):
    try:
        pol = obj["labels"][0]["polarity"]
        if isinstance(pol, str):
            pol = pol.strip().lower().replace(" ", "").replace("▁", "")
        return pol if pol in ABSA_POLARITY_SET else None
    except Exception:
        return None

def parse_polarity_from_text(raw):
    if not raw:
        return None
    s = raw.lower().replace(" ", "").replace("▁", "")
    for pol in ABSA_POLARITIES:
        if pol in s:
            return pol
    return None

def predict_conditioned_polarity(raw, category):
    obj = extract_first_json(raw)
    pol = parse_polarity_from_obj(obj) if obj else None
    if pol is None:
        pol = parse_polarity_from_text(raw)

    if pol is None:
        return None
    return (category, pol)

In [ ]:
from collections import defaultdict

def macro_f1_polarity(y_true, y_pred, labels=ABSA_POLARITIES):
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)

    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}

        for c, true_pol in true_map.items():
            pred_pol = pred_map.get(c)
            if pred_pol == true_pol:
                tp[true_pol] += 1
            else:
                fn[true_pol] += 1
                if pred_pol is not None:
                    fp[pred_pol] += 1

    f1s = []
    for pol in labels:
        p = tp[pol] / (tp[pol] + fp[pol]) if (tp[pol] + fp[pol]) else 0.0
        r = tp[pol] / (tp[pol] + fn[pol]) if (tp[pol] + fn[pol]) else 0.0
        f1 = (2*p*r)/(p+r) if (p+r) else 0.0
        f1s.append(f1)

    return sum(f1s) / len(f1s)


In [ ]:
def polarity_accuracy(y_true, y_pred):
    correct = total = 0
    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}
        for c, tp in true_map.items():
            total += 1
            if pred_map.get(c) == tp:
                correct += 1
    return correct / total if total else 0.0

In [ ]:
def predict_absa_label(model, tokenizer, prompt: str, category: str):
    raw = generate_chat(model, tokenizer, prompt)
    pred = predict_conditioned_polarity(raw, category)
    return pred, raw

In [ ]:
from tqdm import tqdm

def eval_absa_conditioned(rows, demos=None):
    bad = 0
    y_pred, y_true = [], []

    it = rows

    for r in tqdm(it):
        gold_pairs = [(x["category"], x["polarity"].lower()) for x in r["labels"]]
        gold_cats = sorted({cat for cat, _ in gold_pairs})

        pred_set = set()

        for cat in gold_cats:
            prompt = build_conditioned_absa_prompt(r["text"], cat, demos=demos)

            pred_item, raw = predict_absa_label(model, tokenizer, prompt, cat)

            print("\nRAW:", raw)
            print("PRED:", pred_item)
            print("PROMPT:", prompt)
            print("REAL:", [(x["category"], x["polarity"].lower()) for x in r["labels"]])

            if pred_item is None:
                bad += 1
                continue

            pred_set.add(pred_item)

        y_true.append(set(gold_pairs))
        y_pred.append(pred_set)

    return {
        "macro-f1": macro_f1_polarity(y_true, y_pred),
        "accuracy": polarity_accuracy(y_true, y_pred),
        "invalid": bad,
        "n": len(it),
    }

In [ ]:
zeroS = eval_absa_conditioned(absa_test)
threeS = eval_absa_conditioned(absa_test, demos=demos_3)
fiveS = eval_absa_conditioned(absa_test, demos=demos_5)

#Analysis
Although the model achieves relatively high accuracy (72.1%), the macro-averaged F1 score is considerably lower (0.38). This discrepancy indicates a strong class imbalance and a tendency to over-predict the dominant positive polarity, which is a known limitation of large language models in zero-shot sentiment classification. Performance on minority classes such as neutral and conflict remains challenging.

In [ ]:
results = {
    "zeroS": zeroS,
    "threeS": threeS,
    "fiveS": fiveS
}


In [ ]:
results_path = "drive/MyDrive/FYP/absa/prompt_engineering_results/conditioned_absa_test_eval_resultsV2.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False)

We frame ABSA polarity prediction as a conditioned classification task. For each review and gold aspect category, the model is prompted to predict only the sentiment polarity. The prompt enforces a strict JSON-only output format to enable deterministic parsing. Any outputs that violate the format are counted as invalid predictions. Evaluation is performed using macro-averaged F1 and accuracy across polarity classes.

In [ ]:
from google.colab import runtime
runtime.unassign()